# 06 – Vessel segmentation on the CD31 channel (hysteresis threshold)
This notebook segments blood vessels in the **CD31 channel** of the background-corrected PhenoCycler image from 03a with
**two thresholds (hysteresis)**. Every connected CD31+ structure is one vessel object. The vessel read-outs (area, major and
minor axis, eccentricity, orientation) are those of VeSpA (Grion G et al., bioRxiv 2026, doi:10.64898/2026.06.15.732366).

**Why two thresholds.** Along a vessel the CD31 intensity varies a lot. A single Otsu threshold keeps only the bright parts
and cuts each vessel into fragments.

Hysteresis uses a **high threshold** (Otsu) to find the vessels. It then follows each vessel into its dimmer parts down to
a **low threshold**. The low threshold is the 99.5th percentile of the CD31 signal in tissue farther than `BG_DIST_UM` from
any vessel. Pixels above the low threshold count only if they are connected to a pixel above the high threshold, so dim
background speckle away from vessels is not included.

**Why not "no threshold".** CD31 is a clean marker, but its background-corrected image is not zero outside vessels. The 03a
background surface is a low percentile (20th) of each block, so autofluorescence and camera noise above that surface remain.
The channel statistics of 03a show the share of tissue pixels above zero in column `pct_px_nonzero`; for CD31 it is a large
part of the tissue. "Every signal > 0" would therefore label much of the tumour as vessel.

**Why no dilation and no lumen filling (as in VeSpA).** In the multiplex IF images the capillaries form a dense network. The
dilation merges neighbouring capillaries, and the tumour tissue between them is then filled as "lumen".

**Steps for each sample**
1. Read CD31 from the **raw image** and subtract the same background surface as 03a: the 20th percentile of 256-px
   blocks of tissue, smoothed. Pixels outside the analysis region (QuPath outline) are set to 0.
   The 03a corrected image (`<sample>_bgcorrected.ome.tif`) is not used. 03a sets every channel to 0 inside its artefact
   mask, and that mask also covers ordinary tissue and vessels. In an 800 µm window of ss2 it was 14 % of the tumour
   region, and the raw CD31 and CD34 showed continuous vessels running through those areas. The 03a mask is only used for
   the column `artefact_frac` (share of each vessel object inside it), so real folds or debris can be removed afterwards.
2. Smooth lightly (`SMOOTH_SIGMA_UM`) and scale to 0–255; the `SCALE_PERCENTILE` of the region becomes 255.
3. Compute both thresholds once per sample from the whole region. High: Otsu. Low: the 99.5th percentile of the
   background tissue.
4. Keep connected pixels above the low threshold (8-connectivity) that contain a bright part above the high threshold of at
   least `MIN_SEED_UM2`. Objects smaller than `MIN_OBJECT_UM2` are removed.
5. Measure every object and link it to the 03c cells.

**Reading the results**
- No fragments are joined across gaps without CD31 signal. A connected capillary network gives one object.
- The CD31+ vessel area (% of the region) is therefore the most robust read-out. The object number counts CD31+ structures,
  not strictly vessels.
- No lumens are filled, so the area is the area of the CD31+ endothelium.
- For thin, branched objects the minor axis is not the vessel calibre. `mean_width_um` (area / skeleton length) is the
  better estimate.

**How to use it**
1. Set the paths and `TUNE_SAMPLE`.
2. Run parts 1–2 and check the threshold figure on 2–3 samples. If the outlines leak into the tissue around vessels, raise
   `LOW_PERCENTILE` (e.g. 99.5). If dim vessel parts are still missed, lower it (e.g. 98).
3. Run part 3 for all samples. Keep the settings the same for every sample.
4. To check the outlines in QuPath, drag `<sample>_vessels.geojson` onto the same image.

| Part | Content | Output |
|---|---|---|
| 1 | Settings and functions | – |
| 2 | Threshold check on one window of `TUNE_SAMPLE` | `FIG_DIR/<label>_cd31_threshold.png` |
| 3 | All samples | in `<OUTPUT_ROOT>/<sample>/<OUT_NAME>/`: `<sample>_vessels.csv`, `<sample>_vessel_contours.csv`, `<sample>_vessels.geojson`, `<sample>_vessel_labels.tif`, `<sample>_cells_vessels.csv`, `<sample>_vessel_cells.csv`, overview and zoom figures |
| 4 | Summary of all samples | `FIG_DIR/vessels_summary.csv`, `FIG_DIR/vessels_summary.png` |

## 1. Settings and functions

In [ ]:
# ── EDIT ────────────────────────────────────────────────────────────────────────────────────────
OUTPUT_ROOT   = 'path/to/output'      # same as in 03a–03c
SAMPLES       = [f'ss{i}' for i in range(1, 19)]     # samples without 03a output are skipped
TUNE_SAMPLE   = 'ss5'                                # sample for the threshold check (part 2)
TUNE_CROP     = None                                 # (x, y) top-left corner (px) of the check window; None = window with most CD31
TUNE_SIZE_UM  = 500                                  # width of the check window
OUT_NAME      = 'vessels'                            # name of the output folders: <OUTPUT_ROOT>/<OUT_NAME>/ and <OUTPUT_ROOT>/<sample>/<OUT_NAME>/
FIG_DIR       = OUTPUT_ROOT + '/' + OUT_NAME + '/'   # threshold check, summary table and figure of all samples
PIXEL_SIZE_UM = 0.5                                  # µm per pixel

# input
SOURCE        = 'raw'                                # 'raw': raw image + 03a background surface, without the 03a artefact zeroing
                                                     # 'bgcorrected': the 03a output (has holes where 03a found artefacts)
RAW_IMAGE     = 'path/to/raw/{s}.ome.tif'        # raw image of each sample ({s} = ss1 …), column raw_image of samples.csv
BG_BLOCK_PX, BG_QUANTILE, BG_SMOOTH = 256, 20, 1.0   # background surface as in 03a (quantile_surface, block 256 px, 20th pct)

# signal
SIGNAL_CHANNELS  = ['CD31']   # e.g. ['CD31', 'CD34']: the brighter of the scaled channels is used per pixel
SMOOTH_SIGMA_UM  = 0.5        # Gaussian smoothing before the threshold (0 = off); reduces pixel noise
SCALE_PERCENTILE = 99.9       # this percentile of the region pixels becomes 255 in the 8-bit signal
USE_REGION       = True       # True: only inside the 03a analysis region (QuPath outline); False: all tissue
USE_ARTEFACTS    = False      # True: remove the 03a artefact mask. Off by default, because that mask also covers bright
                              # vessels (they are positive in many channels); see 'artefact_frac' in the vessel table instead
ARTEFACT_MIN_UM2 = 2000       # with USE_ARTEFACTS: only artefact areas at least this large are removed (0 = remove all)

# high threshold: finds the vessels (one value per sample, computed from all pixels of the region)
THRESHOLD_MODE  = 'otsu'      # 'otsu', 'percentile', 'value' or 'none' ('none': every pixel with background-corrected
                              #   CD31 > 0 counts, without smoothing, low threshold or seed size)
PERCENTILE      = 95          # 'percentile': pixels above this percentile of the region pixels
THRESHOLD_VALUE = None        # 'value': CD31 intensity of the background-corrected image (same for every sample)

# low threshold: follows each vessel into its dimmer parts (only pixels connected to a pixel above the high threshold)
LOW_MODE        = 'background'  # 'background': LOW_PERCENTILE of the signal in tissue farther than BG_DIST_UM from any
                                #   pixel above the high threshold; 'fraction': LOW_FRACTION x high threshold;
                                #   'value': LOW_VALUE (CD31 intensity); None: high threshold only (no hysteresis)
LOW_PERCENTILE  = 99.5        # 'background': only this share of background-tissue pixels is brighter than the low threshold
BG_DIST_UM      = 20          # 'background': tissue this far from any pixel above the high threshold counts as background
LOW_FRACTION    = 0.5         # 'fraction'
LOW_VALUE       = None        # 'value'

# objects
MIN_OBJECT_UM2  = 10          # smaller CD31+ objects are removed as pixel noise (10 µm² = 40 px; 0 = keep all)
MIN_SEED_UM2    = 5           # a bright part (above the high threshold) must be at least this large to start a vessel object;
                              # stops single bright noise pixels from growing into dim blobs (5 µm² = 20 px; 0 = off)

# tiles and cells
TILE_PX         = 4096        # tile size for smoothing / skeleton / distances; lower it if memory runs out
MARGIN_PX       = 512         # distances to vessels are computed up to this distance (512 px = 256 µm)
PERIVASCULAR_UM = 10          # cells this close to a vessel (and outside it) count as perivascular
ENDO_TOUCH_UM   = 2           # an endothelial cell within this distance of a vessel counts for that vessel
LINK_CELLS      = True        # link the 03c cells (<sample>_adata_annotated.h5ad) to the vessels
ENDOTHELIAL_TYPES = ['Endothelial', 'Endothelial (weak)', 'VWF+']   # 03c cell types counted as endothelial cells

SAMPLE_KEY = {  # CODEX sample -> transcriptomic name (as in notebook 05)
    'ss1': 'S3-P',   'ss2': 'S3-M1',  'ss3': 'S4-P',   'ss4': 'S6-P',   'ss5': 'S7-M2',  'ss6': 'S9-M',
    'ss7': 'S12-P',  'ss8': 'S12-R',  'ss9': 'S12-M1', 'ss10': 'S12-M2', 'ss11': 'S2-P', 'ss12': 'S7-M1',
    'ss13': 'S11-P2', 'ss14': 'S1-M', 'ss15': 'S3-M2', 'ss16': 'S10-P2', 'ss17': 'S10-M', 'ss18': 'S13-M'}
# ────────────────────────────────────────────────────────────────────────────────────────────────

**Thresholds.** Otsu chooses the high threshold from the pixel histogram of each sample, and the low threshold comes from
the background of the same sample, so both adapt to the staining intensity of each slide. A **fixed `THRESHOLD_VALUE`**
(and `LOW_MODE = 'value'`) applies the same CD31 intensities to all samples. Use it if the staining is comparable between
slides and the thresholds printed for each sample differ a lot.

In [ ]:
import os, json, time, warnings, logging
import numpy as np, pandas as pd, tifffile, cv2
import matplotlib, matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from skimage.measure import regionprops_table
from skimage.morphology import skeletonize
from scipy import ndimage as ndi
warnings.filterwarnings('ignore'); logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({'font.family': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 11,
                     'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
                     'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

def save(fig, path, show=True):
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show() if show else plt.close(fig)

SIGMA_PX = SMOOTH_SIGMA_UM / PIXEL_SIZE_UM
MIN_PX = MIN_OBJECT_UM2 / PIXEL_SIZE_UM ** 2
BG_DIST_PX = BG_DIST_UM / PIXEL_SIZE_UM
MIN_SEED_PX = MIN_SEED_UM2 / PIXEL_SIZE_UM ** 2
if THRESHOLD_MODE == 'none':                     # no threshold: CD31 > 0 in the background-corrected image, nothing else
    SIGMA_PX, MIN_SEED_PX, LOW_MODE = 0.0, 0.0, None
print(f'threshold: {THRESHOLD_MODE}, smoothing sigma {SIGMA_PX:g} px, minimum object {MIN_PX:g} px, low threshold: {LOW_MODE}')

**Input.** The CD31 plane is read from the 03a image. The region and artefact masks come from `<sample>_masks_lowres.npz`
(4× down-sampled). The scaling and the thresholds are computed once from every 8th pixel of the region. Both masks
(above low, above high) are built tile by tile, with an overlap of the smoothing radius. They are then labelled for the
whole slide at once, so no object is cut at a tile border.

In [ ]:
def _channel_pages(tif, n):
    for cand in (list(tif.pages), [p for s in tif.series for p in s.pages], list(tif.series[0].pages) if tif.series else []):
        if len(cand) == n:
            return cand
    raise AssertionError(f'cannot find {n} channel pages')

def read_plane(path, names, name):
    with tifffile.TiffFile(path) as tif:
        return _channel_pages(tif, len(names))[names.index(name)].asarray()

def tiles(H, W, t=TILE_PX):
    for y in range(0, H, t):
        for x in range(0, W, t):
            yield y, min(y + t, H), x, min(x + t, W)

def window_mask(lr, ds, y0, y1, x0, x1):
    """Full-resolution window of a low-resolution mask."""
    ly0, lx0 = y0 // ds, x0 // ds
    m = lr[ly0:-(-y1 // ds), lx0:-(-x1 // ds)].repeat(ds, 0).repeat(ds, 1)
    m = m[y0 - ly0 * ds:, x0 - lx0 * ds:][:y1 - y0, :x1 - x0]
    out = np.zeros((y1 - y0, x1 - x0), bool); out[:m.shape[0], :m.shape[1]] = m
    return out

def maxpool(a, f):
    """Down-sample by the maximum of f x f blocks (keeps thin vessels visible), row strip by row strip."""
    h, w = a.shape[0] // f, a.shape[1] // f
    out = np.zeros((h, w), a.dtype)
    for i in range(0, h, 256):
        j = min(i + 256, h)
        out[i:j] = a[i * f:j * f, :w * f].reshape(j - i, f, w, f).max(axis=(1, 3))
    return out

def pool_mean(plane, f, h, w):
    """Mean of f x f blocks -> (h, w) array (strip by strip, as pool() in 03a)."""
    out = np.zeros((h, w), np.float32)
    hh, ww = min(h, plane.shape[0] // f), min(w, plane.shape[1] // f)
    for i in range(0, hh, 256):
        j = min(i + 256, hh)
        out[i:j, :ww] = plane[i * f:j * f, :ww * f].astype(np.float32).reshape(j - i, f, ww, f).mean(axis=(1, 3))
    return out

def background_grid(plane, tissue_lr, ds):
    """03a 'quantile_surface': BG_QUANTILE-th percentile of the tissue in BG_BLOCK_PX blocks of the ds-pooled image, empty
    blocks filled from their neighbours, smoothed. Returns (grid, block size in full-resolution px)."""
    small = pool_mean(plane, ds, *tissue_lr.shape)
    b = max(1, BG_BLOCK_PX // ds)
    nby, nbx = -(-small.shape[0] // b), -(-small.shape[1] // b)
    pad = np.full((nby * b, nbx * b), np.nan, np.float32)
    pad[:small.shape[0], :small.shape[1]] = np.where(tissue_lr, small, np.nan)
    v = pad.reshape(nby, b, nbx, b).transpose(0, 2, 1, 3).reshape(nby, nbx, b * b)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        grid = np.nanpercentile(v, BG_QUANTILE, axis=2)
    grid[np.isfinite(v).mean(axis=2) < 0.10] = np.nan
    w = np.isfinite(grid).astype(np.float32)
    if w.sum() == 0:
        return np.zeros_like(grid, np.float32), b * ds
    g = np.nan_to_num(grid).astype(np.float32); out = np.full(g.shape, np.nan, np.float32)
    for sg in (max(BG_SMOOTH, 0.5), 4 * max(BG_SMOOTH, 0.5) + 4, 50.0):
        num = ndi.gaussian_filter(g * w, sg, mode='nearest'); den = ndi.gaussian_filter(w, sg, mode='nearest')
        todo = np.isnan(out) & (den > 1e-3); out[todo] = (num / np.maximum(den, 1e-6))[todo]
    out[np.isnan(out)] = np.nanmedian(grid)
    return out, b * ds

def surface_window(grid, block, y0, y1, x0, x1):
    """Bilinear interpolation of the background grid for one window (same as upsample_grid in 03a)."""
    def axis(a0, a1, n_in):
        c = np.clip((np.arange(a0, a1) + 0.5) / block - 0.5, 0, n_in - 1)
        i0 = np.floor(c).astype(np.int64); return i0, np.minimum(i0 + 1, n_in - 1), (c - i0).astype(np.float32)
    ya, yb, wy = axis(y0, y1, grid.shape[0]); xa, xb, wx = axis(x0, x1, grid.shape[1])
    gx = grid[:, xa] * (1 - wx) + grid[:, xb] * wx
    return gx[ya] * (1 - wy)[:, None] + gx[yb] * wy[:, None]

def channel_window(P, i, y0, y1, x0, x1):
    """Channel i of a window, background subtracted (raw image) or as written by 03a."""
    a = P['planes'][i][y0:y1, x0:x1].astype(np.float32)
    if P['bgs'][i] is not None:
        a -= surface_window(*P['bgs'][i], y0, y1, x0, x1)
        np.clip(a, 0, None, out=a)
    return a

def signal_window(S, y0, y1, x0, x1, valid=None):
    """CD31 signal on a 0-255 scale (float): smoothed, scaled with the sample scale, brightest channel, 0 outside the region.
    Pixels outside the region are set to 0 before smoothing, so bright artefacts cannot bleed into the region."""
    sig = None
    for i, sc in enumerate(S['scales']):
        a = channel_window(S, i, y0, y1, x0, x1)
        if valid is not None:
            a[~valid] = 0
        if SIGMA_PX > 0:
            a = cv2.GaussianBlur(a, (0, 0), SIGMA_PX)
        a = np.clip(a * (255.0 / sc), 0, 255)
        sig = a if sig is None else np.maximum(sig, a)
    if valid is not None:
        sig[~valid] = 0
    return sig

def prepare(s):
    """Read the CD31 plane(s) and masks of one sample, compute the scale and the threshold."""
    base = os.path.join(OUTPUT_ROOT, s, s + '_')
    cfg = json.load(open(base + 'run_config.json'))
    names, failed = cfg['channel_names'], set(cfg.get('FAILED_MARKERS', []))
    chans = [c for c in SIGNAL_CHANNELS if c in names and c not in failed]
    if not chans:
        raise ValueError(f'none of {SIGNAL_CHANNELS} is available (failed in this sample: {sorted(failed & set(SIGNAL_CHANNELS))})')
    m = np.load(base + 'masks_lowres.npz'); ds = int(m['bg_ds'])
    if SOURCE == 'raw':
        planes = [read_plane(RAW_IMAGE.format(s=s), names, c) for c in chans]
        bg_tissue = m['tissue_lr'].astype(bool) & ~m['art_lr'].astype(bool)          # as in 03a
        bgs = [background_grid(p, bg_tissue, ds) for p in planes]
    else:
        planes = [read_plane(base + 'bgcorrected.ome.tif', names, c) for c in chans]
        bgs = [None] * len(chans)
    P = dict(planes=planes, bgs=bgs)
    H, W = planes[0].shape
    region_lr = (m['region_lr'] if USE_REGION and 'region_lr' in m else m['tissue_lr']).astype(bool)
    px_um2 = (ds * PIXEL_SIZE_UM) ** 2
    art_all = m['art_lr'].astype(bool) & region_lr           # 03a artefact mask inside the region
    art_lr = np.zeros_like(region_lr)                        # the part that is removed
    if USE_ARTEFACTS:
        n_a, lab_a, st_a, _ = cv2.connectedComponentsWithStats(m['art_lr'].astype(np.uint8), connectivity=8)
        big = st_a[:, cv2.CC_STAT_AREA] * px_um2 >= ARTEFACT_MIN_UM2; big[0] = False
        art_lr = big[lab_a] & region_lr
    valid_lr = region_lr & ~art_lr
    region_mm2 = valid_lr.sum() * (ds * PIXEL_SIZE_UM) ** 2 / 1e6
    # smoothed values of every 8th pixel in the region -> scale per channel and thresholds. Pixels in the 03a artefact mask are
    # left out here (so folds cannot shift the thresholds) and kept apart to report how much bright CD31 lies in that mask
    stats_lr = valid_lr & ~art_all
    vals, art_vals = [[] for _ in chans], [[] for _ in chans]
    for y0, y1, x0, x1 in tiles(H, W):
        vf = window_mask(stats_lr, ds, y0, y1, x0, x1)
        va = window_mask(art_all, ds, y0, y1, x0, x1)
        if not (vf[::8, ::8].any() or va[::8, ::8].any()): continue
        for i in range(len(planes)):
            for keep, out in ((vf, vals[i]), (va, art_vals[i])):
                if not keep[::8, ::8].any(): continue
                a = channel_window(P, i, y0, y1, x0, x1); a[~keep] = 0
                if SIGMA_PX > 0: a = cv2.GaussianBlur(a, (0, 0), SIGMA_PX)
                out.append(a[::8, ::8][keep[::8, ::8]])
    vals = [np.concatenate(v) for v in vals]
    art_vals = [np.concatenate(v) if v else np.zeros(0, np.float32) for v in art_vals]
    scales = [max(float(np.percentile(v, SCALE_PERCENTILE)), 1e-6) for v in vals]
    sv = np.max([np.clip(v * (255.0 / sc), 0, 255) for v, sc in zip(vals, scales)], axis=0)
    s8 = sv.astype(np.uint8)
    if THRESHOLD_MODE == 'none':
        thr8 = 0.0
    elif THRESHOLD_MODE == 'otsu':
        thr8, _ = cv2.threshold(s8.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif THRESHOLD_MODE == 'percentile':
        thr8 = float(np.percentile(s8, PERCENTILE))
    elif THRESHOLD_MODE == 'value':
        assert THRESHOLD_VALUE is not None, "THRESHOLD_MODE = 'value' needs THRESHOLD_VALUE"
        thr8 = THRESHOLD_VALUE * 255.0 / scales[0]
    else:
        raise ValueError(THRESHOLD_MODE)
    thr8 = float(np.clip(thr8, 0, 254))
    # low threshold
    pad = int(np.ceil(4 * SIGMA_PX)) + 2
    if LOW_MODE == 'background':
        bg = []
        for y0, y1, x0, x1 in tiles(H, W):
            wy0, wy1 = max(y0 - int(BG_DIST_PX) - pad, 0), min(y1 + int(BG_DIST_PX) + pad, H)
            wx0, wx1 = max(x0 - int(BG_DIST_PX) - pad, 0), min(x1 + int(BG_DIST_PX) + pad, W)
            vf = window_mask(stats_lr, ds, wy0, wy1, wx0, wx1)
            if not vf[y0 - wy0:y1 - wy0, x0 - wx0:x1 - wx0].any(): continue
            sig = signal_window(dict(P, scales=scales), wy0, wy1, wx0, wx1, vf)
            far = cv2.distanceTransform((sig <= thr8).astype(np.uint8), cv2.DIST_L2, 5) > BG_DIST_PX
            c = (slice(y0 - wy0, y1 - wy0, 8), slice(x0 - wx0, x1 - wx0, 8))
            bg.append(sig[c][(far & vf)[c]])
        low8 = float(np.percentile(np.concatenate(bg), LOW_PERCENTILE))
    elif LOW_MODE == 'fraction':
        low8 = LOW_FRACTION * thr8
    elif LOW_MODE == 'value':
        assert LOW_VALUE is not None, "LOW_MODE = 'value' needs LOW_VALUE"
        low8 = LOW_VALUE * 255.0 / scales[0]
    elif LOW_MODE is None:
        low8 = thr8
    else:
        raise ValueError(LOW_MODE)
    low8 = min(low8, thr8)
    a8 = np.max([np.clip(v * (255.0 / sc), 0, 255) for v, sc in zip(art_vals, scales)], axis=0) if len(art_vals[0]) else np.zeros(0)
    n_hi, n_hi_art = int((sv > thr8).sum()), int((a8 > thr8).sum())
    pct_bright_in_art = 100 * n_hi_art / max(n_hi + n_hi_art, 1)
    S = dict(sample=s, base=base, cfg=cfg, chans=chans, planes=planes, bgs=bgs, H=H, W=W, ds=ds, valid_lr=valid_lr,
             region_lr=region_lr, art_lr=art_lr, art_all=art_all, pct_bright_in_art=pct_bright_in_art,
             region_mm2=region_mm2, scales=scales, thr8=thr8, thr_int=thr8 * scales[0] / 255.0,
             low8=low8, low_int=low8 * scales[0] / 255.0,
             pct_above=100 * float((sv > thr8).mean()), label=SAMPLE_KEY.get(s, s.upper()))
    print(f"{s} ({S['label']}): {SOURCE} image, {H} x {W} px, channels {chans}, region {region_mm2:.2f} mm², "
          f"scale {', '.join(f'{c}={v:.0f}' for c, v in zip(chans, scales))} | high ({THRESHOLD_MODE}): "
          f"{thr8:.0f}/255 = {S['thr_int']:.0f} {chans[0]} intensity, {S['pct_above']:.1f} % of region pixels above | "
          f"low ({LOW_MODE}): {low8:.1f}/255 = {S['low_int']:.0f} {chans[0]} intensity (low/high = {low8 / max(thr8, 1e-6):.2f})")
    print(f"   03a artefact mask in the region: {art_all.sum() * px_um2 / 1e6:.3f} mm², "
          f"{'removed: ' + format(art_lr.sum() * px_um2 / 1e6, '.3f') + ' mm²' if USE_ARTEFACTS else 'not removed'}; "
          f"{pct_bright_in_art:.1f} % of the bright CD31 pixels lie in it")
    return S

def threshold_window(S, y0, y1, x0, x1):
    """Masks (bool) of a window: signal above the low and above the high threshold, inside the region."""
    sig = signal_window(S, y0, y1, x0, x1, window_mask(S['valid_lr'], S['ds'], y0, y1, x0, x1))
    return sig > S['low8'], sig > S['thr8']

def components(binary):
    """Connected components (8-connectivity): labels (int32), number of labels incl. background, area per label."""
    binary = binary.astype(np.uint8)
    if binary.size < 2 ** 31 - 1:
        n, lab, st, _ = cv2.connectedComponentsWithStats(binary, connectivity=8, ltype=cv2.CV_32S)
        return lab, n, st[:, cv2.CC_STAT_AREA].astype(np.int64)
    lab, n = ndi.label(binary, structure=np.ones((3, 3), bool), output=np.int32); n += 1   # very large slides
    area = np.zeros(n, np.int64)
    for i in range(0, lab.shape[0], 512):
        area += np.bincount(lab[i:i + 512].ravel(), minlength=n)
    return lab, n, area

def label_objects(low, high, min_px=MIN_PX, min_seed_px=MIN_SEED_PX):
    """Hysteresis: connected pixels above the low threshold (8-connectivity) that contain a bright part (connected pixels
    above the high threshold, at least min_seed_px) -> labels 1..n; objects smaller than min_px are removed."""
    high = high.astype(bool)
    if min_seed_px > 1:
        hl, hn, ha = components(high)
        good = ha >= min_seed_px; good[0] = False
        high = good[hl]; del hl
    lab, n, area = components(low)
    seeded = np.zeros(n, bool); seeded[np.unique(lab[high])] = True
    keep = (area >= min_px) & seeded; keep[0] = False
    lut = np.zeros(n, np.int32); lut[keep] = np.arange(1, keep.sum() + 1, dtype=np.int32)
    for i in range(0, lab.shape[0], 512):
        lab[i:i + 512] = lut[lab[i:i + 512]]
    return lab, int(keep.sum())

def outlines(m):
    """Outer outline and holes of one object (pixel mask) as point arrays."""
    cs, hier = cv2.findContours(m.astype(np.uint8), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    outer = [i for i in range(len(cs)) if hier[0][i][3] == -1]
    o = max(outer, key=lambda i: cv2.contourArea(cs[i]))
    holes = [cs[i][:, 0, :] for i in range(len(cs))                 # holes smaller than MIN_OBJECT_UM2 are not drawn
             if hier[0][i][3] == o and len(cs[i]) >= 3 and cv2.contourArea(cs[i]) >= max(MIN_PX, 1)]
    return cs[o][:, 0, :], holes

## 2. Threshold check on one window
The figure shows one window of `TUNE_SAMPLE`:
- A: the CD31 signal; the 03a artefact mask is outlined in red. If red outlines sit on vessels, keep `USE_ARTEFACTS = False`.
- B: the objects with the high (Otsu) threshold alone.
- C: the pixels added by the low threshold (orange).
- D: the final objects.
The window is the one with the most CD31+ pixels, unless `TUNE_CROP` gives its corner. Check 2–3 samples by
changing `TUNE_SAMPLE`.

In [ ]:
def _lowres_signal(S, f):
    """8-bit signal down-sampled by the block maximum (for overviews and the window choice)."""
    H, W = S['H'], S['W']
    out = np.zeros((H // f, W // f), np.uint8)
    for y0, y1, x0, x1 in tiles(H, W):
        y1c, x1c = y0 + (y1 - y0) // f * f, x0 + (x1 - x0) // f * f
        if y1c <= y0 or x1c <= x0: continue
        v = window_mask(S['valid_lr'], S['ds'], y0, y1c, x0, x1c)
        out[y0 // f:y1c // f, x0 // f:x1c // f] = maxpool(signal_window(S, y0, y1c, x0, x1c, v), f).astype(np.uint8)
    return out

def densest_window(S, size):
    """Top-left corner (px) of the window with the most above-threshold pixels."""
    f = 8
    hit = (_lowres_signal(S, f) > S['thr8']).astype(np.float32)
    k = max(1, size // f)
    score = cv2.boxFilter(hit, -1, (k, k), normalize=False, anchor=(0, 0), borderType=cv2.BORDER_CONSTANT)
    score = score[:max(1, hit.shape[0] - k + 1), :max(1, hit.shape[1] - k + 1)]   # windows that fit in the image
    y, x = np.unravel_index(np.argmax(score), score.shape)
    return int(x * f), int(y * f)

def draw_outlines(ax, lab, color='lime', lw=0.7, offset=(0, 0)):
    segs = []
    for sl, l in zip(ndi.find_objects(lab), range(1, lab.max() + 1)):
        if sl is None: continue
        o, holes = outlines(lab[sl] == l)
        for q in [o] + holes:
            segs.append(np.vstack([q, q[:1]]) + [sl[1].start + offset[0], sl[0].start + offset[1]])
    ax.add_collection(LineCollection(segs, colors=color, linewidths=lw))

def threshold_figure(S, x0=None, y0=None, size_um=TUNE_SIZE_UM, path=None):
    size = int(round(size_um / PIXEL_SIZE_UM))
    if x0 is None:
        x0, y0 = densest_window(S, size)
    x1, y1 = min(x0 + size, S['W']), min(y0 + size, S['H'])
    pad = int(np.ceil(4 * SIGMA_PX)) + 2                  # same smoothing as in the whole-slide run
    wy0, wy1, wx0, wx1 = max(y0 - pad, 0), min(y1 + pad, S['H']), max(x0 - pad, 0), min(x1 + pad, S['W'])
    c = (slice(y0 - wy0, y1 - wy0), slice(x0 - wx0, x1 - wx0))
    sig = signal_window(S, wy0, wy1, wx0, wx1, window_mask(S['valid_lr'], S['ds'], wy0, wy1, wx0, wx1))[c]
    sig_all = signal_window(S, wy0, wy1, wx0, wx1, window_mask(S['region_lr'], S['ds'], wy0, wy1, wx0, wx1))[c]
    art = window_mask(S['art_all'], S['ds'], wy0, wy1, wx0, wx1)[c]
    lab, n = label_objects(sig > S['low8'], sig > S['thr8'])
    high_only, n_high = label_objects(sig > S['thr8'], sig > S['thr8'])
    mask = np.zeros(sig.shape + (3,), np.float32)                      # white: above high, orange: added by the low threshold
    mask[lab > 0] = (1.0, 0.55, 0.0); mask[(lab > 0) & (sig > S['thr8'])] = (1, 1, 1)
    fig, axs = plt.subplots(2, 2, figsize=(13, 13.6), constrained_layout=True); axs = axs.ravel()
    axs[0].imshow(sig_all, cmap='gray', vmin=0, vmax=255)
    if art.any(): axs[0].contour(art, [0.5], colors='red', linewidths=1)
    axs[0].set_title(f'A  {"/".join(S["chans"])} signal; red: 03a artefact mask ({"removed" if USE_ARTEFACTS else "not removed"})', loc='left')
    axs[1].imshow(sig, cmap='gray', vmin=0, vmax=255); draw_outlines(axs[1], high_only, color='magenta')
    axs[1].set_title(f'B  high threshold only ({THRESHOLD_MODE}, {S["thr8"]:.0f}): {n_high} objects' if THRESHOLD_MODE != 'none'
                     else f'B  every pixel with CD31 > 0: {n_high} objects', loc='left')
    axs[2].imshow(mask, interpolation='nearest')
    axs[2].set_title(f'C  above high (white) + connected above low ({S["low8"]:.1f}, orange)' if THRESHOLD_MODE != 'none'
                     else 'C  mask (CD31 > 0)', loc='left')
    axs[3].imshow(sig, cmap='gray', vmin=0, vmax=255); draw_outlines(axs[3], lab)
    axs[3].set_title(f'D  vessel objects ≥ {MIN_OBJECT_UM2:g} µm²: {n}', loc='left')
    bar = 100 / PIXEL_SIZE_UM
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(0, sig.shape[1]); ax.set_ylim(sig.shape[0], 0)
        ax.plot([sig.shape[1] * 0.95 - bar, sig.shape[1] * 0.95], [sig.shape[0] * 0.95] * 2, color='white', lw=3)
    fig.suptitle(f"{S['label']} ({S['sample']}) | CD31 {'hysteresis threshold' if THRESHOLD_MODE != 'none' else 'without threshold (> 0)'}, window x={x0}, y={y0} px, {size_um} µm, scale bar 100 µm",
                 fontsize=13, fontweight='bold')
    save(fig, path or FIG_DIR + f"{S['label']}_cd31_threshold.png")
    return x0, y0

S_tune = prepare(TUNE_SAMPLE)
TUNE_CROP = threshold_figure(S_tune, *(TUNE_CROP or (None, None)))   # to check a vessel you saw in QuPath, set TUNE_CROP to its (x, y) in px
print('window corner (x, y) =', TUNE_CROP, '-> set TUNE_CROP to this value to keep the window')

## 3. All samples
For each sample the cell below writes these files to `<OUTPUT_ROOT>/<sample>/<OUT_NAME>/`:

| File | Content |
|---|---|
| `<sample>_vessels.csv` | one row per vessel object (see below) |
| `<sample>_vessel_contours.csv` | outer outline points in the VeSpA format (`contour_id`, `point_order`, `x`, `y`) |
| `<sample>_vessels.geojson` | outlines (with holes) as QuPath detections carrying the measurements; drag onto the image in QuPath |
| `<sample>_vessel_labels.tif` | label image at full resolution (0 = no vessel, otherwise `vessel_id`) |
| `<sample>_cells_vessels.csv` | for every 03c cell: the `vessel_id` it lies in, the vessel within `PERIVASCULAR_UM` (`perivascular_vessel_id`), the nearest vessel and the distance to it |
| `<sample>_vessel_cells.csv` | number of cells of each type inside and around each vessel |
| `<label>_vessels_overview.png`, `<label>_vessels_zoom.png` | quality control |

**Columns of `<sample>_vessels.csv`**
- `x`, `y`: centroid in px, in the same coordinates as the cells.
- `area_um2`: CD31+ area.
- `major_axis_um`, `minor_axis_um`, `eccentricity`, `orientation`: as in VeSpA.
- `perimeter_um`, `solidity`.
- `skeleton_length_um`: approximate length of the centre line.
- `mean_width_um`: area divided by skeleton length.
- `mean_CD31`: mean background-corrected intensity.
- `n_cells_in`, `n_endothelial`: counts from 03c.
- `artefact_frac`: share of the object inside the 03a artefact mask. Objects near 1 are worth checking in QuPath.

`dist_to_vessel_um` is empty for cells farther than `MARGIN_PX` from any vessel.

In [ ]:
def segment_sample(S):
    """Low and high masks of the whole sample (tile by tile, overlapping by the smoothing radius), then hysteresis at once."""
    H, W = S['H'], S['W']
    pad = int(np.ceil(4 * SIGMA_PX)) + 2
    low = np.zeros((H, W), np.uint8); high = np.zeros((H, W), bool)
    for cy0, cy1, cx0, cx1 in tiles(H, W):
        if not window_mask(S['valid_lr'], S['ds'], cy0, cy1, cx0, cx1).any():
            continue
        wy0, wy1, wx0, wx1 = max(cy0 - pad, 0), min(cy1 + pad, H), max(cx0 - pad, 0), min(cx1 + pad, W)
        lo, hi = threshold_window(S, wy0, wy1, wx0, wx1)
        c = (slice(cy0 - wy0, cy1 - wy0), slice(cx0 - wx0, cx1 - wx0))
        low[cy0:cy1, cx0:cx1] = lo[c]; high[cy0:cy1, cx0:cx1] = hi[c]
    labels, n = label_objects(low, high)
    S['pct_high_area'] = 100 * high.sum() * PIXEL_SIZE_UM ** 2 / (S['region_mm2'] * 1e6)   # area above the high threshold only
    del low, high
    return labels, n


def measure(S, labels, n):
    H, W = labels.shape
    t = pd.DataFrame(regionprops_table(labels, properties=('label', 'area', 'centroid', 'axis_major_length', 'axis_minor_length',
                                                           'eccentricity', 'orientation', 'perimeter', 'solidity')))
    # mean intensity and skeleton length, strip by strip / tile by tile
    sums, skel, in_art = np.zeros(n + 1), np.zeros(n + 1, np.int64), np.zeros(n + 1, np.int64)
    for i in range(0, H, 512):
        sums += np.bincount(labels[i:i + 512].ravel(), weights=channel_window(S, 0, i, min(i + 512, H), 0, W).ravel(), minlength=n + 1)
        a = window_mask(S['art_all'], S['ds'], i, min(i + 512, H), 0, W)
        if a.any(): in_art += np.bincount(labels[i:i + 512][a], minlength=n + 1)
    pad = 32
    for cy0, cy1, cx0, cx1 in tiles(H, W):
        wy0, wy1, wx0, wx1 = max(cy0 - pad, 0), min(cy1 + pad, H), max(cx0 - pad, 0), min(cx1 + pad, W)
        win = labels[wy0:wy1, wx0:wx1]
        if not win.any(): continue
        sk = skeletonize(win > 0)[cy0 - wy0:cy1 - wy0, cx0 - wx0:cx1 - wx0]
        skel += np.bincount(labels[cy0:cy1, cx0:cx1][sk], minlength=n + 1)
    lab = t['label'].to_numpy()
    V = pd.DataFrame({'vessel_id': lab, 'x': t['centroid-1'], 'y': t['centroid-0'], 'area_px': t['area'],
                      'area_um2': t['area'] * PIXEL_SIZE_UM ** 2,
                      'major_axis_um': t['axis_major_length'] * PIXEL_SIZE_UM, 'minor_axis_um': t['axis_minor_length'] * PIXEL_SIZE_UM,
                      'eccentricity': t['eccentricity'], 'orientation': t['orientation'],
                      'perimeter_um': t['perimeter'] * PIXEL_SIZE_UM, 'solidity': t['solidity'],
                      'skeleton_length_um': np.maximum(skel[lab], 1) * PIXEL_SIZE_UM,
                      f"mean_{S['chans'][0]}": sums[lab] / t['area'].to_numpy()})
    V['mean_width_um'] = V.area_um2 / V.skeleton_length_um
    V['artefact_frac'] = in_art[lab] / t['area'].to_numpy()       # share of the object inside the 03a artefact mask
    polys = []
    for sl, l in zip(ndi.find_objects(labels), range(1, n + 1)):
        o, holes = outlines(labels[sl] == l)
        off = [sl[1].start, sl[0].start]
        polys.append((o + off, [h + off for h in holes]))
    return V, polys


def link_cells(S, labels):
    """Vessel each 03c cell lies in, nearest vessel within PERIVASCULAR_UM, and distance to the nearest vessel."""
    import anndata as ad
    f = S['base'] + 'adata_annotated.h5ad'
    if not os.path.exists(f):
        print('  no annotated cells:', f); return None, None
    a = ad.read_h5ad(f, backed='r')
    assert {'x', 'y', 'cell_type'} <= set(a.obs.columns), f'{f} has no x / y / cell_type in obs'
    C = pd.DataFrame({'x': a.obs['x'].to_numpy(float), 'y': a.obs['y'].to_numpy(float),
                      'cell_type': a.obs['cell_type'].astype(str).to_numpy()}, index=a.obs_names)
    a.file.close()
    H, W = labels.shape
    xi = np.clip(np.round(C.x.to_numpy()).astype(int), 0, W - 1)
    yi = np.clip(np.round(C.y.to_numpy()).astype(int), 0, H - 1)
    C['vessel_id'] = labels[yi, xi]
    C['perivascular_vessel_id'] = 0
    C['nearest_vessel_id'] = 0
    C['dist_to_vessel_um'] = np.nan
    ring = PERIVASCULAR_UM / PIXEL_SIZE_UM
    for cy0, cy1, cx0, cx1 in tiles(H, W):
        idx = np.flatnonzero((yi >= cy0) & (yi < cy1) & (xi >= cx0) & (xi < cx1))
        if not len(idx): continue
        wy0, wy1, wx0, wx1 = max(cy0 - MARGIN_PX, 0), min(cy1 + MARGIN_PX, H), max(cx0 - MARGIN_PX, 0), min(cx1 + MARGIN_PX, W)
        win = labels[wy0:wy1, wx0:wx1]
        if not win.any(): continue
        d, (ny, nx) = ndi.distance_transform_edt(win == 0, return_indices=True)
        yy, xx = yi[idx] - wy0, xi[idx] - wx0
        dist = d[yy, xx]
        near = win[ny[yy, xx], nx[yy, xx]]
        C.iloc[idx, C.columns.get_loc('dist_to_vessel_um')] = np.where(dist <= MARGIN_PX, dist * PIXEL_SIZE_UM, np.nan)
        C.iloc[idx, C.columns.get_loc('perivascular_vessel_id')] = np.where((dist > 0) & (dist <= ring), near, 0)
        C.iloc[idx, C.columns.get_loc('nearest_vessel_id')] = np.where(dist <= MARGIN_PX, near, 0)
    long = pd.concat([C[C.vessel_id > 0].groupby(['vessel_id', 'cell_type']).size().rename('n').reset_index().assign(zone='inside'),
                      C[C.perivascular_vessel_id > 0].groupby(['perivascular_vessel_id', 'cell_type']).size().rename('n')
                      .reset_index().rename(columns={'perivascular_vessel_id': 'vessel_id'}).assign(zone='perivascular')])
    return C, long[['vessel_id', 'zone', 'cell_type', 'n']]


try:
    from shapely.geometry import shape as _shape, mapping as _mapping
except ImportError:                       # without shapely, outlines are written as traced
    _shape = None

def write_geojson(path, V, polys, S):
    keys = ['vessel_id', 'area_um2', 'major_axis_um', 'minor_axis_um', 'eccentricity', 'mean_width_um', f"mean_{S['chans'][0]}"]
    keys += [k for k in ('n_endothelial', 'n_cells_in') if k in V]
    ring = lambda q: q.astype(float).tolist() + [q[0].astype(float).tolist()]
    feats = []
    for (_, r), (o, holes) in zip(V.iterrows(), polys):
        geom = {'type': 'Polygon', 'coordinates': [ring(o)] + [ring(h) for h in holes]}
        if _shape is not None and not _shape(geom).is_valid:      # pixel outlines can touch themselves at one corner
            geom = _mapping(_shape(geom).buffer(0))
        feats.append({'type': 'Feature', 'geometry': geom,
                      'properties': {'objectType': 'detection', 'name': f"vessel {int(r.vessel_id)}",
                                     'classification': {'name': 'Vessel', 'color': [0, 200, 0]},
                                     'measurements': {k: float(r[k]) for k in keys}}})
    json.dump({'type': 'FeatureCollection', 'features': feats}, open(path, 'w'))


def overview_figure(S, V, labels, path):
    f = max(1, int(np.ceil(max(S['H'], S['W']) / 2500)))
    sig = _lowres_signal(S, f)
    ves = maxpool((labels > 0).astype(np.uint8), f)
    rgb = np.dstack([sig, sig, sig]).astype(np.float32) / 255
    rgb[ves > 0] = 0.4 * rgb[ves > 0] + 0.6 * np.array([0, 1, 0])
    fig, ax = plt.subplots(figsize=(10, 10 * sig.shape[0] / sig.shape[1]))
    ax.imshow(rgb, interpolation='nearest')
    reg = cv2.resize(S['valid_lr'].astype(np.uint8), (sig.shape[1], sig.shape[0]), interpolation=cv2.INTER_NEAREST)
    ax.contour(reg, [0.5], colors='yellow', linewidths=0.6)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{S['label']} ({S['sample']}) | CD31 (grey), vessel objects (green), analysis region (yellow); {len(V)} objects", loc='left')
    save(fig, path, show=False)


def zoom_figure(S, V, labels, path, n=3, size_um=TUNE_SIZE_UM):
    size = int(round(size_um / PIXEL_SIZE_UM))
    if V.empty: return
    g = V.assign(gx=(V.x // size).astype(int), gy=(V.y // size).astype(int)).groupby(['gx', 'gy']).area_px.sum()
    fig, axs = plt.subplots(1, n, figsize=(5.2 * n, 5.4))
    for ax, ((gx, gy), a) in zip(np.atleast_1d(axs), g.sort_values(ascending=False).head(n).items()):
        x0, y0 = gx * size, gy * size; x1, y1 = min(x0 + size, S['W']), min(y0 + size, S['H'])
        v = window_mask(S['valid_lr'], S['ds'], y0, y1, x0, x1)
        ax.imshow(signal_window(S, y0, y1, x0, x1, v), cmap='gray', vmin=0, vmax=255)
        draw_outlines(ax, labels[y0:y1, x0:x1].copy())
        ax.set_xlim(0, x1 - x0); ax.set_ylim(y1 - y0, 0); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'x={x0}, y={y0} px', fontsize=10)
    fig.suptitle(f"{S['label']} ({S['sample']}) | vessel objects (green) on {'/'.join(S['chans'])}, {size_um} µm windows",
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); save(fig, path, show=False)


summary = []
for s in SAMPLES:
    src = RAW_IMAGE.format(s=s) if SOURCE == 'raw' else os.path.join(OUTPUT_ROOT, s, s + '_bgcorrected.ome.tif')
    if not os.path.exists(os.path.join(OUTPUT_ROOT, s, s + '_run_config.json')) or not os.path.exists(src):
        print(f'{s}: no 03a output or image not found ({src}), skipped'); continue
    t0 = time.time()
    try:
        S = S_tune if s == TUNE_SAMPLE else prepare(s)
    except ValueError as e:
        print(f'{s}: skipped -', e); continue
    labels, n = segment_sample(S)
    if n == 0:
        print(f'{s}: no CD31+ objects found'); continue
    V, polys = measure(S, labels, n)
    out = os.path.join(OUTPUT_ROOT, s, OUT_NAME) + os.sep; os.makedirs(out, exist_ok=True)
    C, long = link_cells(S, labels) if LINK_CELLS else (None, None)
    if C is not None:
        V['n_cells_in'] = V.vessel_id.map(C[C.vessel_id > 0].groupby('vessel_id').size()).fillna(0).astype(int)
        endo = C[C.cell_type.isin(ENDOTHELIAL_TYPES)]
        touch = pd.concat([endo.loc[endo.vessel_id > 0, 'vessel_id'],
                           endo.loc[(endo.perivascular_vessel_id > 0) & (endo.dist_to_vessel_um <= ENDO_TOUCH_UM), 'perivascular_vessel_id']])
        V['n_endothelial'] = V.vessel_id.map(touch.value_counts()).fillna(0).astype(int)
        C.to_csv(out + f'{s}_cells_vessels.csv')
        long.to_csv(out + f'{s}_vessel_cells.csv', index=False)
    V.insert(0, 'sample', s)
    V.to_csv(out + f'{s}_vessels.csv', index=False)
    pd.DataFrame([(i, j, x, y) for i, (o, _) in zip(V.vessel_id, polys) for j, (x, y) in enumerate(o)],
                 columns=['contour_id', 'point_order', 'x', 'y']).to_csv(out + f'{s}_vessel_contours.csv', index=False)
    write_geojson(out + f'{s}_vessels.geojson', V, polys, S)
    tifffile.imwrite(out + f'{s}_vessel_labels.tif', labels, compression='zlib', tile=(512, 512))
    overview_figure(S, V, labels, out + f"{S['label']}_vessels_overview.png")
    zoom_figure(S, V, labels, out + f"{S['label']}_vessels_zoom.png")
    row = dict(sample=s, label=S['label'], channels='/'.join(S['chans']), high_threshold_8bit=round(S['thr8'], 1),
               high_threshold_intensity=round(S['thr_int'], 1), low_threshold_8bit=round(S['low8'], 1),
               low_threshold_intensity=round(S['low_int'], 1), region_mm2=S['region_mm2'], n_objects=len(V),
               objects_per_mm2=len(V) / S['region_mm2'], vessel_area_pct=100 * V.area_um2.sum() / (S['region_mm2'] * 1e6),
               high_only_area_pct=S['pct_high_area'], pct_bright_cd31_in_03a_artefact_mask=round(S['pct_bright_in_art'], 2),
               median_area_um2=V.area_um2.median(), median_mean_width_um=V.mean_width_um.median())
    if 'n_endothelial' in V:
        row['pct_objects_with_endothelial_cell'] = 100 * (V.n_endothelial > 0).mean()
        row['pct_area_with_endothelial_cell'] = 100 * V.area_um2[V.n_endothelial > 0].sum() / V.area_um2.sum()
    summary.append(row)
    print(f"  {s}: {len(V)} objects, {row['objects_per_mm2']:.1f}/mm², CD31+ vessel area {row['vessel_area_pct']:.2f} % of the region, "
          f"(high threshold alone: {row['high_only_area_pct']:.2f} %), median width {row['median_mean_width_um']:.1f} µm  "
          f"({time.time() - t0:.0f} s)\n    files: {out}")
    del labels, V, polys
    if s != TUNE_SAMPLE: del S

## 4. Summary of all samples
- **Vessel area:** summed CD31+ object area in % of the analysis region.
- **Objects:** number of CD31+ objects per mm² of the region.
- **Median width:** median `mean_width_um` of the objects.

In [ ]:
SUM = pd.DataFrame(summary)
SUM.to_csv(FIG_DIR + 'vessels_summary.csv', index=False)
display(SUM.round(2)) if 'display' in globals() else print(SUM.round(2).to_string())
if len(SUM):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (col, t) in zip(axs, [('vessel_area_pct', 'CD31+ vessel area (% of region)'), ('objects_per_mm2', 'CD31+ objects per mm²'),
                                  ('median_mean_width_um', 'Median object width (µm)')]):
        ax.bar(SUM.label, SUM[col], color='#2a9d55'); ax.set_title(t); ax.tick_params(axis='x', rotation=90)
    plt.tight_layout(); save(fig, FIG_DIR + 'vessels_summary.png')

## 5. Remove artefacts you drew in QuPath
Use this part after the first run (parts 1–4), when you have found artefacts in QuPath, such as folds or out-of-focus
areas at the tissue edge.
1. In QuPath, open the **raw image** of the sample. Draw annotations around the artefacts. Closed shapes (polygon,
   brush, ellipse) and lines drawn around an area (polyline, freehand line) both work; a line is closed from its end
   back to its start.
2. Use **File › Export objects as GeoJSON** and save the file as `<OUTPUT_ROOT>/<sample>/<sample>_artefacts.geojson`
   (e.g. `path/to/output/ss2/ss2_artefacts.geojson`). Only **annotations** are used from this file.
   Detections, such as the vessel outlines you loaded for checking, are ignored. An annotation that covers more than half
   of the tumour region is also ignored, because it is most likely the tumour outline.
3. Run the two cells below. Every sample in `SAMPLES` that has such a file is processed; the others are left as they are.
4. In QuPath, delete the old vessel detections and drag in the new `<sample>_vessels.geojson`. Check that the artefacts
   are gone. If not, change the annotations, export again and re-run the cells below.

**What the cells do for each sample**
- Vessel objects with at least `EXCLUDE_MIN_OVERLAP` (default 50 %) of their area inside the drawn areas are removed.
  All other objects stay exactly as in the first run, with the same `vessel_id`, outline and measurements.
- The drawn areas are subtracted from the tumour region area, so vessel area (%) and objects per mm² refer to the tissue
  without the artefacts.
- The 03c cells are linked again to the remaining vessels. Cells inside the drawn areas get `in_artefact_area = True`.
- The files in `<sample>/<OUT_NAME>/` are rewritten, and the rows of these samples in `vessels_summary.csv` are updated.
  `high_only_area_pct` is left empty for them, because it belongs to the first run.
- The files of the first run are kept in `<sample>/<OUT_NAME>/before_artefact_removal/`. Every re-run starts again from
  these files, so you can change the annotations and run the cells again as often as needed.
- The printout says how many annotations were used and how many objects were ignored. If no annotation is usable, the
  first-run results are restored.

After a kernel restart, run parts 1 and 2 first; they define the functions used here. Part 3 does not need to be run again.

In [ ]:
# ── EDIT ────────────────────────────────────────────────────────────────────────────────────────
ARTEFACT_GEOJSON    = OUTPUT_ROOT + '/{s}/{s}_artefacts.geojson'   # QuPath annotations around artefacts ({s} = ss1 …)
EXCLUDE_MIN_OVERLAP = 0.5      # a vessel object is removed if at least this share of its area lies inside the drawn areas
# ────────────────────────────────────────────────────────────────────────────────────────────────
import shutil

def read_annotations(path):
    """Polygons of the ANNOTATIONS in a QuPath GeoJSON export, one list of (ring, is_outer) per annotation.
    Detections (e.g. the vessel outlines loaded from <sample>_vessels.geojson) are ignored."""
    g = json.load(open(path))
    feats = g['features'] if isinstance(g, dict) and 'features' in g else (g if isinstance(g, list) else [g])
    def rings(geom):
        t = geom.get('type') if isinstance(geom, dict) else None
        if t == 'GeometryCollection':
            for gg in geom.get('geometries', []):
                yield from rings(gg)
        elif t in ('Polygon', 'MultiPolygon'):
            for poly in ([geom['coordinates']] if t == 'Polygon' else geom['coordinates']):
                for k, ring in enumerate(poly):
                    yield np.asarray(ring, float), k == 0
        elif t in ('LineString', 'MultiLineString'):        # lines drawn around an area are closed (end joined to start)
            for line in ([geom['coordinates']] if t == 'LineString' else geom['coordinates']):
                if len(line) >= 3:
                    yield np.asarray(line, float), True
    anns, n_skipped = [], 0
    for f in feats:
        if not isinstance(f, dict): continue
        props = f.get('properties') or {}
        cls = props.get('classification')
        cls = cls.get('name', '') if isinstance(cls, dict) else str(cls or '')
        if str(props.get('objectType', 'annotation')).lower() != 'annotation' or cls == 'Vessel':
            n_skipped += 1; continue
        r = list(rings(f.get('geometry', f)))
        if r: anns.append(r)
    return anns, n_skipped

def rasterise(anns, shape, ds=1):
    """Mask of the annotations on a grid down-sampled by ds (holes inside annotations are left out)."""
    m = np.zeros(shape, np.uint8)
    for ann in anns:
        for ring, outer in ann:
            cv2.fillPoly(m, [np.round(ring / ds).astype(np.int32).reshape(-1, 1, 2)], 1 if outer else 0)
    return m.astype(bool)

def relink_cells(base, labels):
    """The same as link_cells in part 3: vessel each 03c cell lies in, vessel within PERIVASCULAR_UM, nearest vessel, distance."""
    import anndata as ad
    f = base + 'adata_annotated.h5ad'
    if not os.path.exists(f):
        return None, None
    a = ad.read_h5ad(f, backed='r')
    C = pd.DataFrame({'x': a.obs['x'].to_numpy(float), 'y': a.obs['y'].to_numpy(float),
                      'cell_type': a.obs['cell_type'].astype(str).to_numpy()}, index=a.obs_names)
    a.file.close()
    H, W = labels.shape
    xi = np.clip(np.round(C.x.to_numpy()).astype(int), 0, W - 1)
    yi = np.clip(np.round(C.y.to_numpy()).astype(int), 0, H - 1)
    C['vessel_id'] = labels[yi, xi]
    C['perivascular_vessel_id'] = 0; C['nearest_vessel_id'] = 0; C['dist_to_vessel_um'] = np.nan
    ring = PERIVASCULAR_UM / PIXEL_SIZE_UM
    for cy0, cy1, cx0, cx1 in tiles(H, W):
        idx = np.flatnonzero((yi >= cy0) & (yi < cy1) & (xi >= cx0) & (xi < cx1))
        if not len(idx): continue
        wy0, wy1, wx0, wx1 = max(cy0 - MARGIN_PX, 0), min(cy1 + MARGIN_PX, H), max(cx0 - MARGIN_PX, 0), min(cx1 + MARGIN_PX, W)
        win = labels[wy0:wy1, wx0:wx1]
        if not win.any(): continue
        d, (ny, nx) = ndi.distance_transform_edt(win == 0, return_indices=True)
        yy, xx = yi[idx] - wy0, xi[idx] - wx0
        dist = d[yy, xx]; near = win[ny[yy, xx], nx[yy, xx]]
        C.iloc[idx, C.columns.get_loc('dist_to_vessel_um')] = np.where(dist <= MARGIN_PX, dist * PIXEL_SIZE_UM, np.nan)
        C.iloc[idx, C.columns.get_loc('perivascular_vessel_id')] = np.where((dist > 0) & (dist <= ring), near, 0)
        C.iloc[idx, C.columns.get_loc('nearest_vessel_id')] = np.where(dist <= MARGIN_PX, near, 0)
    long = pd.concat([C[C.vessel_id > 0].groupby(['vessel_id', 'cell_type']).size().rename('n').reset_index().assign(zone='inside'),
                      C[C.perivascular_vessel_id > 0].groupby(['perivascular_vessel_id', 'cell_type']).size().rename('n')
                      .reset_index().rename(columns={'perivascular_vessel_id': 'vessel_id'}).assign(zone='perivascular')])
    return C, long[['vessel_id', 'zone', 'cell_type', 'n']]

def remove_drawn_artefacts(s, geo):
    base = os.path.join(OUTPUT_ROOT, s, s + '_')
    out = os.path.join(OUTPUT_ROOT, s, OUT_NAME) + os.sep
    keep = out + 'before_artefact_removal' + os.sep
    if not os.path.exists(out + f'{s}_vessels.csv') and not os.path.isdir(keep):
        print(f'{s}: no results of part 3 in {out}, skipped'); return None
    if not os.path.isdir(keep):                                  # keep the first-run files once
        os.makedirs(keep)
        for f in os.listdir(out):
            if os.path.isfile(out + f): shutil.copy2(out + f, keep + f)
    # tumour region (as in part 1) and the annotations to use
    m = np.load(base + 'masks_lowres.npz'); ds = int(m['bg_ds'])
    region_lr = (m['region_lr'] if USE_REGION and 'region_lr' in m else m['tissue_lr']).astype(bool)
    if USE_ARTEFACTS:                                            # same removal of 03a artefacts as in the first run
        n_a, lab_a, st_a, _ = cv2.connectedComponentsWithStats(m['art_lr'].astype(np.uint8), connectivity=8)
        big = st_a[:, cv2.CC_STAT_AREA] * (ds * PIXEL_SIZE_UM) ** 2 >= ARTEFACT_MIN_UM2; big[0] = False
        region_lr = region_lr & ~big[lab_a]
    anns, n_det = read_annotations(geo)
    use = [a for a in anns if (rasterise([a], region_lr.shape, ds) & region_lr).sum() <= 0.5 * region_lr.sum()]
    n_big = len(anns) - len(use)                                 # e.g. the tumour outline exported by mistake
    print(f"{s}: {len(use)} artefact annotation(s) used from {os.path.basename(geo)}"
          + (f"; ignored: {n_det} detection(s) such as vessel outlines" if n_det else '')
          + (f"; ignored: {n_big} annotation(s) covering more than half of the tumour region (tumour outline?)" if n_big else ''))
    # start from the first-run files (with no usable annotation, this restores the first-run results)
    V = pd.read_csv(keep + f'{s}_vessels.csv')
    labels = tifffile.imread(keep + f'{s}_vessel_labels.tif')
    H, W = labels.shape
    drawn = rasterise(use, (H, W))
    n_lab = int(labels.max()) + 1
    inside = np.zeros(n_lab, np.int64)
    for i in range(0, H, 512):
        d = drawn[i:i + 512]
        if d.any(): inside += np.bincount(labels[i:i + 512][d], minlength=n_lab)
    frac = inside[V.vessel_id.to_numpy()] / V.area_px.to_numpy()
    drop = frac >= EXCLUDE_MIN_OVERLAP
    alive = np.ones(n_lab, bool); alive[V.vessel_id.to_numpy()[drop]] = False
    for i in range(0, H, 512):
        labels[i:i + 512] *= alive[labels[i:i + 512]]
    removed = V[drop]; V = V[~drop].copy()
    # region area without the drawn areas
    drawn_lr = rasterise(use, region_lr.shape, ds)
    px_mm2 = (ds * PIXEL_SIZE_UM) ** 2 / 1e6
    excluded_mm2 = (region_lr & drawn_lr).sum() * px_mm2
    region_mm2 = (region_lr & ~drawn_lr).sum() * px_mm2
    # cells
    C, long = relink_cells(base, labels) if LINK_CELLS else (None, None)
    if C is not None:
        xi = np.clip(np.round(C.x.to_numpy()).astype(int), 0, W - 1); yi = np.clip(np.round(C.y.to_numpy()).astype(int), 0, H - 1)
        C['in_artefact_area'] = drawn[yi, xi]
        V['n_cells_in'] = V.vessel_id.map(C[C.vessel_id > 0].groupby('vessel_id').size()).fillna(0).astype(int)
        endo = C[C.cell_type.isin(ENDOTHELIAL_TYPES)]
        touch = pd.concat([endo.loc[endo.vessel_id > 0, 'vessel_id'],
                           endo.loc[(endo.perivascular_vessel_id > 0) & (endo.dist_to_vessel_um <= ENDO_TOUCH_UM), 'perivascular_vessel_id']])
        V['n_endothelial'] = V.vessel_id.map(touch.value_counts()).fillna(0).astype(int)
        C.to_csv(out + f'{s}_cells_vessels.csv')
        long.to_csv(out + f'{s}_vessel_cells.csv', index=False)
    # vessel files (outlines are taken from the first run)
    ids = set(V.vessel_id.tolist())
    V.to_csv(out + f'{s}_vessels.csv', index=False)
    cont = pd.read_csv(keep + f'{s}_vessel_contours.csv')
    cont[cont.contour_id.isin(ids)].to_csv(out + f'{s}_vessel_contours.csv', index=False)
    gj = json.load(open(keep + f'{s}_vessels.geojson'))
    vrow = V.set_index('vessel_id')
    feats = []
    for f in gj['features']:
        vid = int(f['properties']['measurements']['vessel_id'])
        if vid in ids:
            for k in f['properties']['measurements']:
                if k in vrow.columns: f['properties']['measurements'][k] = float(vrow.at[vid, k])
            feats.append(f)
    json.dump({'type': 'FeatureCollection', 'features': feats}, open(out + f'{s}_vessels.geojson', 'w'))
    tifffile.imwrite(out + f'{s}_vessel_labels.tif', labels, compression='zlib', tile=(512, 512))
    # figure: what was removed
    S = prepare(s)
    f = max(1, int(np.ceil(max(H, W) / 2500)))
    sig = _lowres_signal(S, f)
    rgb = np.dstack([sig, sig, sig]).astype(np.float32) / 255
    kept_lr = maxpool((labels > 0).astype(np.uint8), f) > 0
    gone_lr = maxpool((np.isin(tifffile.imread(keep + f'{s}_vessel_labels.tif'), removed.vessel_id.to_numpy())).astype(np.uint8), f) > 0
    rgb[kept_lr] = 0.4 * rgb[kept_lr] + 0.6 * np.array([0, 1, 0])
    rgb[gone_lr] = 0.4 * rgb[gone_lr] + 0.6 * np.array([1, 0, 1])
    fig, ax = plt.subplots(figsize=(10, 10 * sig.shape[0] / sig.shape[1]))
    ax.imshow(rgb, interpolation='nearest')
    ax.contour(cv2.resize(drawn_lr.astype(np.uint8), (sig.shape[1], sig.shape[0]), interpolation=cv2.INTER_NEAREST), [0.5], colors='red', linewidths=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{S['label']} ({s}) | drawn artefact areas (red), removed objects (magenta), kept objects (green)", loc='left')
    save(fig, out + f"{S['label']}_vessels_artefacts_removed.png")
    print(f"  {len(removed)} of {len(removed) + len(V)} objects removed ({removed.area_um2.sum() / 1e6:.4f} mm² CD31+ area), "
          f"{excluded_mm2:.3f} mm² of the region excluded; now {len(V)} objects, "
          f"CD31+ vessel area {100 * V.area_um2.sum() / (region_mm2 * 1e6):.2f} % of {region_mm2:.2f} mm²\n    files: {out}")
    row = dict(sample=s, region_mm2=region_mm2, n_objects=len(V), objects_per_mm2=len(V) / region_mm2,
               vessel_area_pct=100 * V.area_um2.sum() / (region_mm2 * 1e6), high_only_area_pct=np.nan,
               median_area_um2=V.area_um2.median(), median_mean_width_um=V.mean_width_um.median(),
               excluded_artefact_mm2=excluded_mm2, n_objects_removed_artefacts=len(removed))
    if 'n_endothelial' in V:
        row['pct_objects_with_endothelial_cell'] = 100 * (V.n_endothelial > 0).mean()
        row['pct_area_with_endothelial_cell'] = 100 * V.area_um2[V.n_endothelial > 0].sum() / V.area_um2.sum()
    return row

In [ ]:
assert 'prepare' in globals() and '_lowres_signal' in globals(), 'run parts 1 and 2 first'
rows = []
for s in SAMPLES:
    geo = ARTEFACT_GEOJSON.format(s=s)
    if os.path.exists(geo):
        r = remove_drawn_artefacts(s, geo)
        if r: rows.append(r)
if not rows:
    print('no artefact GeoJSON found, e.g.', ARTEFACT_GEOJSON.format(s=SAMPLES[0]))
else:
    SUM = pd.read_csv(FIG_DIR + 'vessels_summary.csv')
    for r in rows:
        for k in r:
            if k not in SUM.columns: SUM[k] = np.nan
        SUM.loc[SUM['sample'] == r['sample'], list(r)] = list(r.values())
    SUM.to_csv(FIG_DIR + 'vessels_summary.csv', index=False)
    display(SUM.round(2)) if 'display' in globals() else print(SUM.round(2).to_string())
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (col, t) in zip(axs, [('vessel_area_pct', 'CD31+ vessel area (% of region)'), ('objects_per_mm2', 'CD31+ objects per mm²'),
                                  ('median_mean_width_um', 'Median object width (µm)')]):
        ax.bar(SUM.label, SUM[col], color='#2a9d55'); ax.set_title(t); ax.tick_params(axis='x', rotation=90)
    plt.tight_layout(); save(fig, FIG_DIR + 'vessels_summary.png')

## 6. Update the cells after a new annotation in 03c
Use this part when the cell types of a sample were changed in 03c after this notebook had been run, for example when a
cluster was split into macrophages and monocytes. The vessels are **not** segmented again: the vessel objects, the
artefact removal of part 5 and all vessel measurements stay exactly as they are. For each sample in `REANNOTATED`:
- the cells of the new `<sample>_adata_annotated.h5ad` are linked again to the current vessel objects, in the same way
  as in part 3 (`vessel_id`, `perivascular_vessel_id`, `nearest_vessel_id`, `dist_to_vessel_um`);
- `in_artefact_area` (from part 5) is taken over from the previous cell table, matched by cell ID;
- `<sample>_cells_vessels.csv` and `<sample>_vessel_cells.csv` are rewritten; the cell counts per vessel (`n_cells_in`,
  `n_endothelial`) are updated in `<sample>_vessels.csv` and `<sample>_vessels.geojson`, and the endothelial columns of
  the sample in `vessels_summary.csv`;
- the previous files are kept in `<sample>/<OUT_NAME>/before_reannotation/` (only the first time, so this folder always
  holds the state before the first re-annotation).

The printout compares the number of cells per cell type before and after. After a kernel restart, run the three cells of
part 1 first; parts 2–5 are not needed. Afterwards, run notebook 07 again.

Running part 5 again later is also fine: it starts from the first-run files and links the cells of the current 03c file.

In [ ]:
# ── EDIT ────────────────────────────────────────────────────────────────────────────────────────
REANNOTATED = ['ss6', 'ss14', 'ss18']      # samples whose cell types were changed in 03c
# ────────────────────────────────────────────────────────────────────────────────────────────────
import shutil, anndata as ad

def link_cells_again(base, labels):
    """The same linking as in part 3, for the cells of the current <sample>_adata_annotated.h5ad."""
    a = ad.read_h5ad(base + 'adata_annotated.h5ad', backed='r')
    C = pd.DataFrame({'x': a.obs['x'].to_numpy(float), 'y': a.obs['y'].to_numpy(float),
                      'cell_type': a.obs['cell_type'].astype(str).to_numpy()}, index=a.obs_names.astype(str))
    a.file.close()
    H, W = labels.shape
    xi = np.clip(np.round(C.x.to_numpy()).astype(int), 0, W - 1)
    yi = np.clip(np.round(C.y.to_numpy()).astype(int), 0, H - 1)
    C['vessel_id'] = labels[yi, xi]
    C['perivascular_vessel_id'] = 0; C['nearest_vessel_id'] = 0; C['dist_to_vessel_um'] = np.nan
    ring = PERIVASCULAR_UM / PIXEL_SIZE_UM
    for cy0, cy1, cx0, cx1 in tiles(H, W):
        idx = np.flatnonzero((yi >= cy0) & (yi < cy1) & (xi >= cx0) & (xi < cx1))
        if not len(idx): continue
        wy0, wy1, wx0, wx1 = max(cy0 - MARGIN_PX, 0), min(cy1 + MARGIN_PX, H), max(cx0 - MARGIN_PX, 0), min(cx1 + MARGIN_PX, W)
        win = labels[wy0:wy1, wx0:wx1]
        if not win.any(): continue
        d, (ny, nx) = ndi.distance_transform_edt(win == 0, return_indices=True)
        yy, xx = yi[idx] - wy0, xi[idx] - wx0
        dist = d[yy, xx]; near = win[ny[yy, xx], nx[yy, xx]]
        C.iloc[idx, C.columns.get_loc('dist_to_vessel_um')] = np.where(dist <= MARGIN_PX, dist * PIXEL_SIZE_UM, np.nan)
        C.iloc[idx, C.columns.get_loc('perivascular_vessel_id')] = np.where((dist > 0) & (dist <= ring), near, 0)
        C.iloc[idx, C.columns.get_loc('nearest_vessel_id')] = np.where(dist <= MARGIN_PX, near, 0)
    long = pd.concat([C[C.vessel_id > 0].groupby(['vessel_id', 'cell_type']).size().rename('n').reset_index().assign(zone='inside'),
                      C[C.perivascular_vessel_id > 0].groupby(['perivascular_vessel_id', 'cell_type']).size().rename('n')
                      .reset_index().rename(columns={'perivascular_vessel_id': 'vessel_id'}).assign(zone='perivascular')])
    return C, long[['vessel_id', 'zone', 'cell_type', 'n']]

def update_cells(s):
    base = os.path.join(OUTPUT_ROOT, s, s + '_')
    out = os.path.join(OUTPUT_ROOT, s, OUT_NAME) + os.sep
    need = [out + f'{s}_vessels.csv', out + f'{s}_vessel_labels.tif', base + 'adata_annotated.h5ad']
    if not all(os.path.exists(f) for f in need):
        print(f'{s}: skipped, missing', [os.path.basename(f) for f in need if not os.path.exists(f)]); return None
    files = [f'{s}_cells_vessels.csv', f'{s}_vessel_cells.csv', f'{s}_vessels.csv', f'{s}_vessels.geojson']
    keep = out + 'before_reannotation' + os.sep
    if not os.path.isdir(keep):
        os.makedirs(keep)
        for f in files:
            if os.path.exists(out + f): shutil.copy2(out + f, keep + f)
    old = None
    if os.path.exists(out + f'{s}_cells_vessels.csv'):
        old = pd.read_csv(out + f'{s}_cells_vessels.csv', dtype={0: str}, low_memory=False)
        old = old.set_index(old.columns[0]); old.index = old.index.astype(str)

    labels = tifffile.imread(out + f'{s}_vessel_labels.tif')       # current objects (after part 5, if it was run)
    C, long = link_cells_again(base, labels)
    if old is not None and 'in_artefact_area' in old:
        prev = old['in_artefact_area'].reindex(C.index)
        if prev.isna().any():
            print(f'  {s}: {prev.isna().sum():,} cells were not in the previous table; they are set to in_artefact_area = False. '
                  'If they could lie in a drawn artefact area, run part 5 for this sample instead.')
        C['in_artefact_area'] = prev.fillna(False).astype(bool)

    V = pd.read_csv(out + f'{s}_vessels.csv')
    V['n_cells_in'] = V.vessel_id.map(C[C.vessel_id > 0].groupby('vessel_id').size()).fillna(0).astype(int)
    endo = C[C.cell_type.isin(ENDOTHELIAL_TYPES)]
    touch = pd.concat([endo.loc[endo.vessel_id > 0, 'vessel_id'],
                       endo.loc[(endo.perivascular_vessel_id > 0) & (endo.dist_to_vessel_um <= ENDO_TOUCH_UM), 'perivascular_vessel_id']])
    V['n_endothelial'] = V.vessel_id.map(touch.value_counts()).fillna(0).astype(int)

    C.to_csv(out + f'{s}_cells_vessels.csv')
    long.to_csv(out + f'{s}_vessel_cells.csv', index=False)
    V.to_csv(out + f'{s}_vessels.csv', index=False)
    if os.path.exists(out + f'{s}_vessels.geojson'):
        gj = json.load(open(out + f'{s}_vessels.geojson'))
        vrow = V.set_index('vessel_id')
        for f in gj['features']:
            ms = f['properties']['measurements']; vid = int(ms['vessel_id'])
            for k in ('n_cells_in', 'n_endothelial'):
                if vid in vrow.index: ms[k] = float(vrow.at[vid, k])
        json.dump(gj, open(out + f'{s}_vessels.geojson', 'w'))

    # what changed
    label = SAMPLE_KEY.get(s, s)
    print(f'{s} ({label}): {len(C):,} cells linked to {len(V):,} vessel objects\n    files: {out}')
    if old is not None:
        same = C.index.intersection(old.index)
        was, now = old.loc[same, 'cell_type'].astype(str), C.loc[same, 'cell_type']
        sw = was != now
        print(f'    {int(sw.sum()):,} cells changed their cell type; {len(old) - len(same):,} cells are no longer in 03c, '
              f'{len(C) - len(same):,} are new')
        if sw.any():
            moves = pd.crosstab(was[sw].rename('before'), now[sw].rename('after'))
            print('    cells that changed type (rows: before, columns: after):')
            display(moves) if 'display' in globals() else print(moves.to_string())
        t = pd.DataFrame({'before': old.cell_type.astype(str).value_counts(), 'after': C.cell_type.value_counts()}).fillna(0).astype(int)
        t['change'] = t.after - t.before
        near = lambda D: D[D.dist_to_vessel_um.fillna(np.inf) <= PERIVASCULAR_UM].cell_type.astype(str).value_counts()
        t[f'before, ≤ {PERIVASCULAR_UM:g} µm of a vessel'] = near(old).reindex(t.index).fillna(0).astype(int)
        t[f'after, ≤ {PERIVASCULAR_UM:g} µm of a vessel'] = near(C).reindex(t.index).fillna(0).astype(int)
        involved = set(was[sw]) | set(now[sw])
        changed = t[(t.change != 0) | t.index.isin(involved)]
        print('    number of cells per cell type:')
        display(changed) if 'display' in globals() else print(changed.to_string())
    row = dict(sample=s, pct_objects_with_endothelial_cell=100 * (V.n_endothelial > 0).mean(),
               pct_area_with_endothelial_cell=100 * V.area_um2[V.n_endothelial > 0].sum() / V.area_um2.sum())
    return row

In [ ]:
assert 'tiles' in globals() and 'MARGIN_PX' in globals(), 'run the three cells of part 1 first'
rows = [r for r in (update_cells(s) for s in REANNOTATED) if r]
if rows and os.path.exists(FIG_DIR + 'vessels_summary.csv'):
    SUM = pd.read_csv(FIG_DIR + 'vessels_summary.csv')
    for r in rows:
        for k in r:
            if k not in SUM.columns: SUM[k] = np.nan
        SUM.loc[SUM['sample'] == r['sample'], list(r)] = list(r.values())
    SUM.to_csv(FIG_DIR + 'vessels_summary.csv', index=False)
    print('updated', FIG_DIR + 'vessels_summary.csv', 'for', ', '.join(r['sample'] for r in rows))